# Data Warehouse vs Data Lake vs Lakehouse 

> 数据架构三大范式的深度对比 — 概念、优缺点、架构设计、使用场景

## 1. 三者的核心定义

### 1.1 Data Warehouse

Data Warehouse 是一种**面向分析的、结构化的集中式数据存储系统**。数据在写入前必须经过 ETL（Extract, Transform, Load）清洗和转换，按照预定义的 schema 组织（schema-on-write）。它针对 OLAP（Online Analytical Processing）查询做了深度优化，提供极快的聚合查询性能。

**典型代表**：Snowflake、Amazon Redshift、Google BigQuery、Azure Synapse Analytics、Teradata

**架构特征**：
- **Storage 和 Compute 可分离或耦合**：传统 warehouse（Teradata、Oracle）是耦合的，现代 cloud warehouse（Snowflake、BigQuery）实现了 storage-compute 分离
- **数据组织方式**：star schema（事实表 + 维度表）、snowflake schema、或 data vault
- **数据格式**：proprietary internal format（各家自己的列式存储格式），用户不直接接触底层文件
- **索引和优化**：columnar storage、materialized views、result caching、clustering keys

<img src='./pic/1_data_warehouse_pro_cons.png' width=700>

**优点：**

- **极致的查询性能**：针对 OLAP 做了深度优化，columnar storage、vectorized execution、result caching 使得聚合查询非常快。尤其是 Snowflake 的 auto-clustering 和 BigQuery 的 slot-based execution，对 ad-hoc 分析极为友好。
- **强一致性和数据质量**：schema-on-write 确保进入 warehouse 的数据都经过验证和转换，数据质量高。ACID transactions 保证读写一致性，不会出现 partial write 的脏数据。
- **成熟的 SQL 生态**：标准 SQL 接口，BI 工具（Tableau、Looker、Power BI）原生集成，business analyst 可以直接使用，不需要写代码。
- **Security 和 governance 完善**：细粒度的 access control（row-level、column-level security）、audit log、data masking 等企业级功能成熟。
- **运维简单**：cloud warehouse 基本是全托管服务，自动 scaling、自动备份、自动优化，运维成本低。

**缺点：**

- **成本高**：按 compute 和 storage 分别计费，大规模数据存储在 warehouse 里比放在 S3 贵很多（Snowflake storage 约 $23/TB/month vs S3 约 $23/TB/month standard，但 warehouse 的 compute 成本远高于 Spark on S3）。长期存储大量历史数据很不经济。
- **数据格式封闭**：数据存储在 proprietary format 中，不能被 Spark、PyTorch 等外部工具直接读取。如果想用 ML framework 处理数据，必须先 export 出来。这造成了数据 silo 和 vendor lock-in。
- **不适合非结构化数据**：warehouse 只能存结构化数据（tables），图片、视频、日志、传感器数据等无法直接存储和处理。
- **ETL 复杂度和延迟**：数据必须先通过 ETL pipeline 清洗转换后才能写入，这增加了开发工作量和数据延迟。从原始数据到可查询状态可能需要几小时。
- **Schema 灵活性差**：schema 变更需要谨慎规划（尤其在大型 star schema 中），加字段、改类型都可能影响下游 pipeline 和 dashboard。

### 1.2 Data Lake

Data Lake 是一种**低成本、大规模的原始数据存储**，能够容纳任意格式的数据（结构化、半结构化、非结构化）。数据以原始形态写入，读取时才施加 schema（schema-on-read）。本质上就是 object storage（S3、ADLS、GCS）上的文件集合，配合 catalog 提供基本的数据发现能力。

**典型代表**：Amazon S3 + Glue Catalog、Azure Data Lake Storage（ADLS）、Hadoop HDFS（传统方案）

**架构特征**：
- **Storage layer**：object storage（S3/ADLS/GCS）或 HDFS
- **数据格式**：open format — Parquet、ORC、Avro、JSON、CSV，甚至图片、视频、日志
- **Catalog**：Hive Metastore、AWS Glue Catalog、Apache Atlas 提供 metadata 管理
- **计算引擎**：Spark、Presto/Trino、Hive、Flink 等直接读取 lake 上的文件

<img src='./pic/1_data_lake_pro_cons.png' width=700>

**优点：**

- **极低的存储成本**：object storage（S3 Standard ~$0.023/GB/month、S3 Glacier ~$0.004/GB/month）是所有存储方案中最便宜的。存 PB 级数据也很经济。
- **格式灵活、全类型支持**：可以存储任何格式的数据 — 结构化（Parquet、CSV）、半结构化（JSON、Avro、Protobuf）、非结构化（图片、音频、视频、PDF）。这使得它成为所有原始数据的统一落地点。
- **Schema-on-read 灵活性**：写入时不需要预定义 schema，数据以原始形态存储。不同的消费者可以按自己的需求解释同一份数据，适合探索性分析。
- **Open format 无 vendor lock-in**：数据以 Parquet、ORC 等 open format 存储，任何引擎（Spark、Trino、Flink、DuckDB、Pandas）都可以直接读取，不依赖特定厂商。
- **ML/AI 友好**：PyTorch、TensorFlow、Scikit-learn 等 ML 框架可以直接从 S3 读取数据，不需要额外的 data movement。
- **可扩展性极强**：object storage 几乎无限扩展，没有容量上限。

**缺点：**

- **没有 ACID transactions**：并发写入可能产生 partial writes、corrupted data。没有 isolation，reader 可能读到写到一半的数据。
- **数据质量无保障**：没有 schema enforcement，任何人可以往任何路径写任何格式的文件。久而久之变成 "data swamp"（数据沼泽）— 数据在那里但没人能用。
- **查询性能差**：没有索引、没有 caching、没有 query optimization。Hive on S3 的性能远不如 Snowflake。缺少 data skipping 和 partition pruning 的智能优化（除非手动管理 Hive-style partitioning）。
- **Governance 和 security 薄弱**：原始 data lake 只有 object-level 的 access control（S3 bucket policy、IAM），没有 row-level / column-level security，没有 data lineage，没有 audit log。
- **管理成本高**：需要手动管理 file layout、partitioning、compaction、metadata catalog、data quality checks。没有自动优化，运维负担重。
- **不支持 UPDATE / DELETE**：object storage 上的文件是 immutable 的，不能直接修改某一行。需要读取整个文件、修改、重写，非常低效。



### 1.3 Lakehouse

Lakehouse 是在 Data Lake 的 object storage 之上，增加一层 **transactional metadata layer（table format）**，从而获得 Data Warehouse 级别的数据管理能力。它同时服务 BI 分析和 ML/AI 工作负载，数据只存一份，不需要在 lake 和 warehouse 之间搬运。

**典型代表**：Databricks（Delta Lake）、Apache Iceberg + Trino/Spark、Apache Hudi、Snowflake（通过 Iceberg 集成）

**架构特征**：
- **Storage layer**：仍然是 object storage 上的 open format 文件（Parquet 为主）
- **Table format**：Delta Lake / Iceberg / Hudi 提供 ACID transactions、schema enforcement、time travel
- **Catalog**：Unity Catalog（Databricks）、Nessie、REST Catalog、AWS Glue
- **计算引擎**：多引擎直接访问 — Spark、Flink、Trino、Dremio、DuckDB

<img src='./pic/1_lakehouse_pros.png' width=700>

**优点：**

- **兼具 warehouse 的可靠性和 lake 的灵活性**：ACID transactions + schema enforcement 确保数据质量，同时数据仍以 open format 存储在廉价的 object storage 上。
- **一份数据服务所有工作负载**：BI（通过 Trino/Spark SQL）、ML（通过 Spark MLlib / PyTorch 直接读 Parquet）、streaming（通过 Flink / Structured Streaming）都读同一份数据，消除 data silo 和 ETL 冗余。
- **Time travel 和 audit**：所有历史变更都有 snapshot，可以回溯任意时间点的数据状态，满足 compliance 和 debug 需求。
- **Schema evolution**：可以安全地 add / drop / rename columns 而不 rewrite 历史数据（尤其 Iceberg 通过 column ID 实现）。
- **Open format、无 lock-in**：数据是 Parquet，metadata 也是 open spec（Iceberg 是 Apache 项目、Delta Lake 已捐赠给 Linux Foundation），可以自由切换引擎。
- **成本优势**：存储成本等于 object storage 成本（S3 价格），远低于把所有数据放 Snowflake。Compute 可以用 Spark on EMR/Databricks 按需启停。
- **支持 streaming 写入**：可以增量追加数据，配合 compaction 策略管理文件大小。

**缺点：**

- **查询性能仍不及顶级 warehouse**：虽然 Lakehouse 的查询性能在持续提升（Photon engine、Velox），但在纯 OLAP 场景下，Snowflake / BigQuery 的查询速度仍然更快，尤其是 concurrent query 和 dashboard 场景。
- **运维复杂度更高**：需要管理 compaction job、snapshot expiration、orphan file cleanup、catalog 选型和维护。不像 Snowflake 那样全自动。
- **生态仍在发展中**：Iceberg / Delta / Hudi 三家 table format 竞争中，标准尚未完全统一。虽然有 UniForm / XTable 等互转方案，但增加了复杂度。
- **BI 工具集成不如 warehouse 成熟**：Tableau / Looker 连 Snowflake 是 native connector，一键配置。连 Lakehouse 可能需要通过 Trino / Spark Thrift Server / Databricks SQL 中转，配置更复杂。
- **Concurrent write 性能**：OCC（optimistic concurrency control）在高并发写入场景下可能频繁冲突和 retry，不如 warehouse 的内置 transaction manager 高效。
- **学习曲线**：团队需要理解 table format 的概念（snapshot、manifest、partition spec、compaction），比直接用 Snowflake 的学习成本高。


## 2. 核心维度对比

| 维度 | Data Warehouse | Data Lake | Lakehouse |
|---|---|---|---|
| **数据类型** | 仅结构化数据 | 结构化 + 半结构化 + 非结构化 | 结构化 + 半结构化（非结构化存 lake 层） |
| **Schema 策略** | Schema-on-write | Schema-on-read | 两者兼备（enforcement + evolution） |
| **ACID Transactions** | 完整支持 | 不支持 | 通过 table format 支持 |
| **存储格式** | Proprietary / 封闭格式 | Open format（Parquet、ORC 等） | Open format（Parquet 为主） |
| **存储成本** | 高（$20-40/TB/month 含 compute） | 极低（$2-23/TB/month） | 低（等于 object storage 成本） |
| **查询性能（OLAP）** | 极快（深度优化） | 慢（无索引无优化） | 较快（持续提升中） |
| **UPDATE / DELETE** | 原生支持 | 不支持（需全文件 rewrite） | 通过 COW / MOR 支持 |
| **Time Travel** | 有限支持（如 Snowflake 90 天） | 不支持 | 完整支持（可配置 retention） |
| **ML/AI 支持** | 差（需 export 数据） | 好（直接读文件） | 好（open format + 直接访问） |
| **Streaming 支持** | 有限（micro-batch ingest） | 原生支持 | 原生支持（增量 append） |
| **Governance** | 成熟（row/column-level security） | 薄弱（object-level ACL） | 发展中（Unity Catalog 等） |
| **Vendor Lock-in** | 高（proprietary format） | 低（open format） | 低（open format + open spec） |
| **运维复杂度** | 低（全托管） | 高（手动管理一切） | 中等（需管理 compaction、expiration 等） |
| **BI 工具集成** | 原生、无缝 | 需要额外配置 | 通过 SQL endpoint 集成，不断改善 |
| **典型延迟** | 分钟级（ETL pipeline） | 秒级（直接写入） | 秒到分钟级（streaming + compaction） |

---



## 3. 架构演进路线

实际企业中，数据架构通常经历以下演进：

### 阶段一：Data Warehouse Only（传统企业）

```text
Source Systems → ETL → Data Warehouse → BI Dashboard
```

适合：数据量小（TB 级）、纯 BI 分析场景、structured data only。
问题：随着数据量增长，成本飙升；无法处理日志、图片等非结构化数据；ML 团队拿不到数据。

### 阶段二：Data Lake + Data Warehouse（Two-Tier）

```text
Source Systems → Data Lake (S3) → ETL → Data Warehouse → BI Dashboard
                     ↓
              ML / Data Science（直接读 lake）
```

这是过去 5-10 年最常见的架构。Data lake 作为所有原始数据的 landing zone，经过清洗后将 curated data 加载到 warehouse 做分析。ML 团队直接从 lake 读数据。

问题：
- **数据冗余**：同一份数据在 lake 和 warehouse 各存一份，成本翻倍。
- **ETL 复杂度**：lake → warehouse 的 ETL pipeline 需要维护，数据不一致风险高。
- **Data staleness**：warehouse 中的数据比 lake 晚几小时，BI 看到的不是最新数据。
- **Data swamp 风险**：lake 中的原始数据如果没有 governance，很快就没人能用。

### 阶段三：Lakehouse（融合架构）

```text
Source Systems → Lakehouse (S3 + Table Format) → BI Dashboard
                         ↓                      → ML Training
                         ↓                      → Streaming Analytics
                         ↓                      → Ad-hoc Query
```

数据只存一份，通过 table format 提供事务性和质量保障，所有消费者直接访问。

优势：消除冗余、降低成本、减少 ETL、统一 governance。

### 阶段四：Lakehouse + Warehouse 混合（现实方案）

```text
Source Systems → Lakehouse (S3 + Iceberg) → 大部分分析 + ML
                         ↓
              Hot / Dashboard Data → Snowflake / BigQuery → 高并发 BI Dashboard
```

现实中很多企业采用混合方案：大部分数据和工作负载在 Lakehouse 上完成，但对于**高并发、低延迟的 dashboard 查询**，仍然会把 curated aggregate data 推到 Snowflake / BigQuery 上，因为这些场景下 warehouse 的性能优势明显。

---



## 4. 使用场景推荐

### 选 Data Warehouse 的场景

- **纯 BI / reporting 场景**：公司主要需求是 dashboard 和 ad-hoc SQL 查询，数据量在 TB 级别，分析师团队为主。
- **高并发 dashboard**：需要支持几百个 concurrent queries（如公司级 Looker dashboard），warehouse 的 caching 和 concurrent scaling 更成熟。
- **强 compliance 需求**：金融、医疗行业需要 row-level security、column masking、细粒度 audit log，warehouse 的 governance 功能更完善。
- **团队技术栈简单**：团队主要是 SQL analyst，不想维护 Spark cluster 和 table format，需要全托管服务。
- **示例**：零售企业做销售报表、金融机构做风控分析 dashboard、SaaS 产品的客户 analytics。

### 选 Data Lake 的场景

- **原始数据归档**：需要低成本存储 PB 级原始数据（日志、传感器数据、clickstream），以备将来分析。
- **非结构化数据存储**：图片、视频、音频、PDF 文档等无法放进 warehouse 的数据。
- **ML / AI 训练数据**：ML 团队需要大量原始数据，直接从 S3 读取，不需要 warehouse 的 SQL 接口。
- **数据探索和实验**：data scientist 需要探索各种原始数据，schema-on-read 提供最大灵活性。
- **成本敏感且容忍复杂度**：预算有限，愿意投入工程师时间管理 data quality 和 file layout。
- **示例**：自动驾驶公司存传感器数据、社交媒体平台存用户上传的图片和视频、IoT 设备数据收集。

### 选 Lakehouse 的场景

- **BI + ML 统一平台**：同一份数据既要给 analyst 做 SQL 分析，又要给 ML engineer 做 feature engineering 和 model training。
- **Streaming + batch 混合**：数据从 Kafka 实时写入，既要支持实时 dashboard，又要支持每日 batch 报表。
- **成本优化**：从 Snowflake 迁移大量历史数据到 S3 + Iceberg，节省存储成本，只在 Snowflake 保留 hot data。
- **避免 vendor lock-in**：需要 open format 和 multi-engine 访问，未来可能切换计算引擎。
- **数据工程团队能力强**：有能力管理 Spark cluster、compaction job、table format 配置的工程团队。
- **中大型数据平台**：数据量 PB 级，需要统一的 data platform 服务多个团队和多种工作负载。
- **示例**：电商平台（用户行为分析 + 推荐模型训练）、金融科技（交易数据分析 + 风控模型）、大型互联网公司的统一数据平台。

---



## 5. 面试常见问题与回答

### Q: 为什么不直接把所有数据放 Data Warehouse？

Data warehouse 的存储和 compute 成本都很高。当数据量达到 PB 级时，存储成本会成为瓶颈。而且 warehouse 只能处理结构化数据，ML 团队需要的原始数据和非结构化数据无法存储。此外，proprietary format 造成 vendor lock-in，切换成本很高。所以大型企业通常把大量数据放在 lake 或 lakehouse 上，只把 hot / curated data 放 warehouse。

### Q: Data Lake 为什么容易变成 Data Swamp？

因为 data lake 缺少三个关键能力：schema enforcement（任何人写任何格式的文件都不会报错）、access control（权限管理粗糙）、data catalog and lineage（数据写进去就找不到了）。没有 governance 的 lake 很快就充斥着格式不一、含义不明、质量未知的文件，变成 "data swamp"。Lakehouse 通过 table format 的 schema enforcement 和 catalog 的 metadata management 来解决这个问题。

### Q: Lakehouse 能完全替代 Data Warehouse 吗？

目前还不能完全替代。Lakehouse 在以下场景仍不如专业 warehouse：高并发 dashboard 查询（几百个 concurrent queries）、sub-second 查询延迟需求、成熟的 BI 工具 native 集成、企业级 governance（虽然 Unity Catalog 在追赶）。现实中更常见的是 Lakehouse + warehouse 混合架构，让各自做各自擅长的事。

### Q: Snowflake 也支持 Iceberg table 了，那它算 warehouse 还是 lakehouse？

现代数据架构的边界正在模糊。Snowflake 通过 Iceberg Tables 允许外部引擎直接读取 S3 上的 Parquet 文件，这是 lakehouse 的特征。BigQuery 通过 BigLake 也做了类似的事。可以说它们正在从纯 warehouse 向 lakehouse hybrid 演进。但核心区别仍在：Snowflake 的主力优化路径还是 proprietary engine + proprietary format，Iceberg 支持是一个补充，性能和功能完整度不如 native table。

---



## 6. 快速回顾 Cheat Sheet

| 问题 | 答案 |
|---|---|
| 最便宜的存储？ | Data Lake（object storage 成本） |
| 最快的 OLAP 查询？ | Data Warehouse（深度优化 + caching） |
| 最适合 ML/AI？ | Lakehouse 或 Data Lake（open format 直接访问） |
| 最强的 governance？ | Data Warehouse（row/column-level security 成熟） |
| 最灵活的数据类型？ | Data Lake（任何格式） |
| ACID transactions？ | Warehouse 原生支持，Lakehouse 通过 table format 支持，Lake 不支持 |
| 最低运维成本？ | Data Warehouse（全托管）|
| 最低 vendor lock-in？ | Data Lake / Lakehouse（open format） |
| 现实最优方案？ | 通常是 Lakehouse + Warehouse 混合架构 |

# Lakehouse 深入实操指南 — Iceberg & Delta Lake

> 架构细节 + 核心操作 + 代码示例 + 生产实践

| Lakehouse Table Format | Creator| Notes| 
| -----| --------| ------| 
| Apache Iceberg| Netflix| best for multi-engine interoperability| 
| Delta Lake| Databricks| strongest Spark integration| 
| Apache Hudi| Uber| strong for streaming ingestion| 

| 维度 | Delta Lake | Apache Iceberg | Apache Hudi |
|---|---|---|---|
| **设计核心重点** | Spark生态集成优先 | 引擎中立与查询性能 | 增量处理与CDC优化 |
| **上手难度** | ★★★☆ 最容易 | ★★★★ 中等偏难 | ★★★★ 中等偏难 |
| **最佳适用场景** | Databricks平台/全Spark团队 | 多引擎混合/大规模分析 | 实时更新/CDC同步 |
| **引擎兼容性** | Spark完美，其他需适配 | 所有主流引擎原生支持 | Spark/Flink优秀，其他一般 |
| **Upsert性能** | ★★★★ 良好 | ★★★★ 良好 | ★★★★★ 专门优化 |
| **查询性能** | ★★★★ Spark上优秀 | ★★★★½ 全引擎优秀 | ★★★ 流式查询优 |
| **分区灵活性** | 传统分区 | 隐藏分区 + 分区演化 | 传统分区 |
| **社区生态** | 商业驱动，Databricks主导 | 社区最活跃，多巨头支持 | 特定场景深耕 |
| **运维复杂度** | 简单（在Databricks） | 中等，需调优 | 中等，自动化高 |
| **长期风险** | 供应商锁定 | 技术碎片化 | 场景局限 |


## 1. Iceberg 深入详解

Apache Iceberg is an open table format for data lakes that enables ACID transactions, schema evolution, and fast analytics on large datasets stored in files like Parquet on object storage (S3, GCS, ADLS).  

Iceberg adds a table layer on top of files.  

```text
Query Engine
   │
   ▼
Iceberg Table Metadata
   │
   ▼
Parquet / ORC / Avro files
   │
   ▼
Object Storage (S3 / GCS / ADLS)
```

### 1.1 Iceberg 的完整架构

Iceberg 的设计哲学是**把 table 的所有状态信息都编码在 metadata 文件中**，不依赖任何外部系统（除了 catalog 做入口指针）。这使得它 engine-agnostic，任何能读 Avro/JSON + Parquet 的引擎都可以理解 Iceberg table。

```text
┌────────────────────────────────────────────────────────────────────┐
│                        Catalog Layer                               │
│   (Hive Metastore / AWS Glue / Nessie / REST Catalog)              │
│   职责：table name → 当前 metadata file 的位置                        │
└──────────────────────────┬─────────────────────────────────────────┘
                           │ 指向
                           ▼
┌────────────────────────────────────────────────────────────────────┐
│                   Metadata File (JSON/Avro)                        │
│   内容：                                                            │
│   - table UUID                                                     │
│   - schema（含 column IDs）                                         │
│   - partition spec（含 transform 定义）                              │
│   - sort order                                                     │
│   - current-snapshot-id                                            │
│   - snapshot log（所有历史 snapshot 的列表）                          │
│   - properties（table 级别配置）                                     │
└──────────────────────────┬─────────────────────────────────────────┘
                           │ current snapshot 指向
                           ▼
┌────────────────────────────────────────────────────────────────────┐
│                   Manifest List (Avro)                             │
│   内容：manifest file 列表 + 每个 manifest 的 partition summary       │
│   - manifest_path                                                  │
│   - manifest_length                                                │
│   - partition_spec_id                                              │
│   - added_snapshot_id                                              │
│   - added/existing/deleted file counts                             │
│   - partition field summaries (contains_null, lower_bound, upper)  │
└─────────┬──────────────────┬──────────────────┬────────────────────┘
          ▼                  ▼                  ▼
┌──────────────┐   ┌──────────────┐   ┌──────────────┐
│ Manifest File│   │ Manifest File│   │ Manifest File│  (Avro)
│              │   │              │   │              │
│ 每条 entry:   │   │              │   │              │
│ - file_path  │   │              │   │              │
│ - partition  │   │              │   │              │
│ - record_cnt │   │              │   │              │
│ - col stats  │   │              │   │              │
│   (min/max)  │   │              │   │              │
│ - status     │   │              │   │              │
│   (ADD/DEL)  │   │              │   │              │
└──────┬───────┘   └──────┬───────┘   └──────┬───────┘
       ▼                  ▼                  ▼
  [data files]       [data files]       [data files]   (Parquet/ORC/Avro)
```

**Catalog 层的 atomic commit 机制**（不同 catalog 实现不同）：

| Catalog | Atomic Commit 实现 |
|---|---|
| Hive Metastore | 表级锁（`LOCK TABLE`）+ metadata location update |
| AWS Glue | Conditional update（`UpdateTable` with version check） |
| Nessie | Git-like commit（compare-and-swap on branch pointer） |
| REST Catalog | Server-side atomic update（Iceberg REST spec） |
| Hadoop Catalog | Atomic rename on HDFS（不支持 S3，因为 S3 rename 不是 atomic） |



### 1.2 Iceberg Table 创建与基础操作

**使用 Spark SQL 创建 Iceberg Table：**

```sql
-- 配置 Spark 使用 Iceberg
-- spark-defaults.conf 或 SparkSession builder 中设置：
-- spark.sql.catalog.iceberg_catalog = org.apache.iceberg.spark.SparkCatalog
-- spark.sql.catalog.iceberg_catalog.type = hive  (或 glue / rest / hadoop)
-- spark.sql.catalog.iceberg_catalog.warehouse = s3://my-bucket/warehouse

-- 创建 namespace（相当于 database）
CREATE NAMESPACE IF NOT EXISTS iceberg_catalog.analytics;

-- 创建表（注意 hidden partitioning 的写法）
CREATE TABLE iceberg_catalog.analytics.events (
    event_id     BIGINT,
    user_id      BIGINT,
    event_type   STRING,
    event_time   TIMESTAMP,
    properties   STRING,        -- JSON string
    amount       DECIMAL(10,2)
)
USING iceberg
PARTITIONED BY (
    day(event_time),            -- hidden partition: 按天
    bucket(16, user_id)         -- hidden partition: user_id 分 16 个桶
)
TBLPROPERTIES (
    'write.format.default' = 'parquet',
    'write.parquet.compression-codec' = 'zstd',
    'write.target-file-size-bytes' = '536870912',   -- 512MB target file size
    'history.expire.max-snapshot-age-ms' = '432000000',  -- 5 天 snapshot retention
    'write.metadata.delete-after-commit.enabled' = 'true',
    'write.metadata.previous-versions-max' = '100'
);
```

**使用 PySpark DataFrame API 创建和写入：**

```python
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("IcebergDemo") \
    .config("spark.sql.catalog.iceberg_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg_catalog.type", "hive") \
    .config("spark.sql.catalog.iceberg_catalog.warehouse", "s3://my-bucket/warehouse") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()

# 方式一：CREATE TABLE AS SELECT (CTAS)
df = spark.read.parquet("s3://raw-data/events/")
df.writeTo("iceberg_catalog.analytics.events") \
  .partitionedBy("day(event_time)", "bucket(16, user_id)") \
  .createOrReplace()

# 方式二：Append 模式写入（生产中最常用）
new_events = spark.read.json("s3://raw-data/events/2024-06-15/")
new_events.writeTo("iceberg_catalog.analytics.events").append()

# 方式三：Dynamic overwrite（只覆盖涉及的 partition）
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
corrected_data.writeTo("iceberg_catalog.analytics.events").overwritePartitions()
```



### 1.3 Schema Evolution 操作

```sql
-- 添加列
ALTER TABLE iceberg_catalog.analytics.events
ADD COLUMNS (
    device_type STRING AFTER event_type,
    session_id  STRING
);

-- 删除列
ALTER TABLE iceberg_catalog.analytics.events DROP COLUMN session_id;

-- 重命名列（安全！基于 column ID，不影响已有 data files）
ALTER TABLE iceberg_catalog.analytics.events RENAME COLUMN properties TO event_metadata;

-- 修改列类型（type promotion）
ALTER TABLE iceberg_catalog.analytics.events
ALTER COLUMN event_id TYPE BIGINT;  -- int -> bigint

-- 重新排序列
ALTER TABLE iceberg_catalog.analytics.events
ALTER COLUMN device_type FIRST;     -- 移到第一列

-- 添加列注释
ALTER TABLE iceberg_catalog.analytics.events
ALTER COLUMN amount COMMENT 'Transaction amount in USD';
```

**PySpark API 方式（使用 Iceberg Java API）：**

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
# 通过 Spark SQL 执行 DDL 是最常用的方式
spark.sql("""
    ALTER TABLE iceberg_catalog.analytics.events
    ADD COLUMNS (country STRING, city STRING)
""")
```



### 1.4 Partition Evolution 操作

```sql
-- 查看当前 partition spec
DESCRIBE TABLE iceberg_catalog.analytics.events;

-- 添加新的 partition field（不影响已有数据）
ALTER TABLE iceberg_catalog.analytics.events
ADD PARTITION FIELD hour(event_time);

-- 替换 partition field
-- 注意：旧数据仍然按旧 spec 分区，新数据按新 spec 分区
ALTER TABLE iceberg_catalog.analytics.events
DROP PARTITION FIELD day(event_time);

ALTER TABLE iceberg_catalog.analytics.events
ADD PARTITION FIELD hour(event_time);

-- 添加 truncate partition
ALTER TABLE iceberg_catalog.analytics.events
ADD PARTITION FIELD truncate(10, event_type);
```

**Partition Evolution 的内部机制：**

```text
Snapshot 1-50:   partition-spec-id = 0   → day(event_time), bucket(16, user_id)
Snapshot 51+:    partition-spec-id = 1   → hour(event_time), bucket(16, user_id)

查询时：
  WHERE event_time = '2024-06-15 14:30:00'
  → 对 spec-0 的 manifest：检查 day = '2024-06-15' 的分区
  → 对 spec-1 的 manifest：检查 hour = '2024-06-15-14' 的分区
  → 两种 spec 的 pruning 都自动完成，用户无感知
```



### 1.5 Time Travel 操作

```sql
-- 按 snapshot ID 查询
SELECT * FROM iceberg_catalog.analytics.events VERSION AS OF 1234567890;

-- 按时间查询
SELECT * FROM iceberg_catalog.analytics.events
TIMESTAMP AS OF '2024-06-15 10:00:00';

-- 查看 snapshot 历史
SELECT * FROM iceberg_catalog.analytics.events.snapshots;
-- 返回：snapshot_id, parent_id, operation, manifest_list, summary, committed_at

-- 查看操作历史
SELECT * FROM iceberg_catalog.analytics.events.history;

-- 查看所有 data files（当前 snapshot）
SELECT * FROM iceberg_catalog.analytics.events.files;

-- 查看所有 manifest files
SELECT * FROM iceberg_catalog.analytics.events.manifests;

-- 查看两个 snapshot 之间的变更（incremental read）
-- 方法一：通过 metadata 表
SELECT * FROM iceberg_catalog.analytics.events.changes
WHERE _change_type IN ('insert', 'delete')
  AND snapshot_id > 100 AND snapshot_id <= 200;

-- 方法二：Spark DataFrame API
spark.read \
    .option("start-snapshot-id", "100") \
    .option("end-snapshot-id", "200") \
    .table("iceberg_catalog.analytics.events")
```

**PySpark 中的 Time Travel：**

```python
# 按 snapshot ID 读取
df_old = spark.read \
    .option("snapshot-id", 1234567890) \
    .table("iceberg_catalog.analytics.events")

# 按时间读取
df_as_of = spark.read \
    .option("as-of-timestamp", "1718438400000") \  # epoch millis
    .table("iceberg_catalog.analytics.events")

# Incremental read（只读增量数据，常用于 CDC 下游消费）
df_incremental = spark.read \
    .option("start-snapshot-id", "100") \
    .option("end-snapshot-id", "200") \
    .table("iceberg_catalog.analytics.events")
```



### 1.6 Rollback 操作

```sql
-- 回滚到指定 snapshot（创建新 snapshot，指向旧的 manifest list）
CALL iceberg_catalog.system.rollback_to_snapshot(
    'analytics.events', 1234567890
);

-- 回滚到指定时间点
CALL iceberg_catalog.system.rollback_to_timestamp(
    'analytics.events', TIMESTAMP '2024-06-15 10:00:00'
);

-- Cherry-pick 某个 snapshot 的变更（把某次 commit 的 changes 应用到当前状态）
CALL iceberg_catalog.system.cherrypick_snapshot(
    'analytics.events', 9876543210
);
```



### 1.7 Compaction 与维护操作

```sql
-- 合并小文件（compaction）
CALL iceberg_catalog.system.rewrite_data_files(
    table => 'analytics.events',
    strategy => 'binpack',    -- 或 'sort'
    options => map(
        'target-file-size-bytes', '536870912',   -- 512MB
        'min-file-size-bytes',   '67108864',     -- 64MB（小于此的才会被 compact）
        'max-file-size-bytes',   '1073741824',   -- 1GB
        'partial-progress.enabled', 'true',       -- 大表分批处理
        'partial-progress.max-commits', '10'
    )
);

-- 带排序的 compaction（提升后续查询的 data skipping 效率）
CALL iceberg_catalog.system.rewrite_data_files(
    table => 'analytics.events',
    strategy => 'sort',
    sort_order => 'event_time ASC, user_id ASC'
);

-- 仅 compact 特定 partition
CALL iceberg_catalog.system.rewrite_data_files(
    table => 'analytics.events',
    where => 'event_time >= TIMESTAMP ''2024-06-01'' AND event_time < TIMESTAMP ''2024-07-01'''
);

-- 合并 manifest files（减少 manifest 数量，加速 planning）
CALL iceberg_catalog.system.rewrite_manifests('analytics.events');

-- 过期旧 snapshot（解除对旧 data files 的引用）
CALL iceberg_catalog.system.expire_snapshots(
    'analytics.events',
    TIMESTAMP '2024-06-10 00:00:00'  -- 该时间之前的 snapshot 全部过期
);

-- 清理 orphan files（删除不被任何 snapshot 引用的文件）
CALL iceberg_catalog.system.remove_orphan_files(
    table => 'analytics.events',
    older_than => TIMESTAMP '2024-06-01 00:00:00',
    dry_run => true  -- 先预览，确认后改为 false
);
```

**生产环境中的维护 DAG（Airflow 示例逻辑）：**

```python
# 典型的 daily maintenance DAG
# Task 1: Compaction（合并小文件）
# Task 2: Rewrite manifests（合并 manifest）
# Task 3: Expire snapshots（过期旧快照，保留 7 天）
# Task 4: Remove orphan files（清理孤儿文件，older_than 3 天）

# 执行顺序：compact → rewrite_manifests → expire_snapshots → remove_orphan_files
# 原因：先 compact 产生新大文件，再 expire 才能释放旧小文件的引用，最后 remove orphan 物理删除
```



### 1.8 Row-Level 操作（UPDATE / DELETE / MERGE）

Iceberg v2 支持 row-level update 和 delete，有两种模式：

**Copy-on-Write（默认）：**

```sql
-- 写入时 rewrite 整个 data file
-- 适合：批量更新、读多写少
SET spark.sql.iceberg.merge-on-read.enabled = false;

-- DELETE
DELETE FROM iceberg_catalog.analytics.events
WHERE event_time < TIMESTAMP '2023-01-01';

-- UPDATE
UPDATE iceberg_catalog.analytics.events
SET amount = amount * 1.1
WHERE event_type = 'purchase' AND event_time >= '2024-06-01';

-- MERGE INTO（upsert / SCD 等场景）
MERGE INTO iceberg_catalog.analytics.events AS target
USING staging_events AS source
ON target.event_id = source.event_id
WHEN MATCHED THEN
    UPDATE SET
        target.event_type = source.event_type,
        target.amount = source.amount
WHEN NOT MATCHED THEN
    INSERT (event_id, user_id, event_type, event_time, amount)
    VALUES (source.event_id, source.user_id, source.event_type,
            source.event_time, source.amount);
```

**Merge-on-Read（v2 特性）：**

```sql
-- 写入时只写 delete file，读取时合并
-- 适合：频繁小批量更新、写多读少
ALTER TABLE iceberg_catalog.analytics.events
SET TBLPROPERTIES (
    'write.delete.mode' = 'merge-on-read',
    'write.update.mode' = 'merge-on-read',
    'write.merge.mode'  = 'merge-on-read'
);

-- 之后的 DELETE / UPDATE / MERGE 操作会生成 delete files 而非 rewrite data files
-- delete file 类型：
--   positional delete: 记录 (file_path, row_position) → 精确标记哪个文件的哪一行被删除
--   equality delete:   记录 (column_values) → 匹配符合条件的行

-- MOR 模式下需要定期 compaction 来合并 delete files 到 data files
CALL iceberg_catalog.system.rewrite_data_files('analytics.events');
```



### 1.9 Branching 和 Tagging（Iceberg 1.2+）

```sql
-- 创建 tag（不可变的 snapshot 引用，常用于发布标记）
ALTER TABLE iceberg_catalog.analytics.events
CREATE TAG `release-2024-q2`
AS OF VERSION 1234567890
RETAIN 365 DAYS;

-- 创建 branch（可变的 snapshot 引用，常用于 ETL staging / audit）
ALTER TABLE iceberg_catalog.analytics.events
CREATE BRANCH `etl-staging`
AS OF VERSION 1234567890
RETAIN 7 DAYS;

-- 在 branch 上写入数据（不影响 main）
spark.conf.set("spark.wap.branch", "etl-staging")
new_data.writeTo("iceberg_catalog.analytics.events").append()

-- 验证 branch 数据
df = spark.read \
    .option("branch", "etl-staging") \
    .table("iceberg_catalog.analytics.events")

-- 确认没问题后，fast-forward merge 到 main
CALL iceberg_catalog.system.fast_forward(
    'analytics.events', 'main', 'etl-staging'
);

-- 删除 branch
ALTER TABLE iceberg_catalog.analytics.events DROP BRANCH `etl-staging`;
```

**Write-Audit-Publish（WAP）模式**：先在 branch 上写入 → 运行数据质量检查 → 通过则 publish 到 main，不通过则 drop branch。这是 Iceberg branch 最重要的生产应用场景。

---



## 2. Delta Lake 深入详解

### 2.1 Delta Lake 的完整架构

Delta Lake 的设计哲学是**用 transaction log 记录表的所有变更**，log 本身就是 table 的 single source of truth。

```text
┌──────────────────────────────────────────────────────────┐
│                    Delta Table 目录结构                   │
│                                                          │
│   s3://my-bucket/warehouse/analytics/events/             │
│   ├── _delta_log/                    ← Transaction log   │
│   │   ├── 00000000000000000000.json  ← Commit 0          │
│   │   ├── 00000000000000000001.json  ← Commit 1          │
│   │   ├── ...                                            │
│   │   ├── 00000000000000000010.checkpoint.parquet        │
│   │   │         ↑ 每 10 个 commit 生成一个 checkpoint      │
│   │   └── _last_checkpoint           ← 指向最新 checkpoint│
│   ├── part-00000-xxx.snappy.parquet  ← Data files        │
│   ├── part-00001-xxx.snappy.parquet                      │
│   └── ...                                                │
└──────────────────────────────────────────────────────────┘
```

**Transaction Log（`_delta_log/`）中每个 JSON 文件包含的 Action 类型：**

| Action | 说明 |
|---|---|
| `add` | 新增一个 data file（包含 path、size、partition values、stats） |
| `remove` | 标记删除一个 data file（逻辑删除，实际文件还在） |
| `metaData` | 更新表的 schema、partition columns、table properties |
| `txn` | Application-level transaction ID（用于 idempotent writes） |
| `protocol` | 读写协议版本（控制哪些 feature 可用） |
| `commitInfo` | Commit metadata（timestamp、operation、user） |
| `domainMetadata` | Domain-specific metadata（Delta 3.0+） |

**Checkpoint 文件的作用：**
- 每 10 个 commit（默认）合并为一个 Parquet 格式的 checkpoint 文件
- Reader 先找到最新 checkpoint，再 replay 之后的 JSON commit
- 避免每次读取都从第 0 个 commit 开始 replay（性能瓶颈）
- `_last_checkpoint` 文件记录最新 checkpoint 的位置



### 2.2 Delta Table 创建与基础操作

**使用 Spark SQL：**

```sql
-- 配置 Spark 使用 Delta Lake
-- spark.sql.extensions = io.delta.sql.DeltaSparkSessionExtension
-- spark.sql.catalog.spark_catalog = org.apache.spark.sql.delta.catalog.DeltaCatalog

-- 创建 Delta Table
CREATE TABLE analytics.events (
    event_id     BIGINT,
    user_id      BIGINT,
    event_type   STRING,
    event_time   TIMESTAMP,
    properties   STRING,
    amount       DECIMAL(10,2)
)
USING delta
PARTITIONED BY (date_trunc('day', event_time))   -- Delta 显式分区（generated column）
LOCATION 's3://my-bucket/warehouse/analytics/events'
TBLPROPERTIES (
    'delta.logRetentionDuration' = '30 days',
    'delta.deletedFileRetentionDuration' = '7 days',
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.targetFileSize' = '536870912',    -- 512MB
    'delta.tuneFileSizesForRewrites' = 'true'
);
```

**使用 PySpark DataFrame API：**

```python
from pyspark.sql import SparkSession
from delta import *

spark = SparkSession.builder \
    .appName("DeltaDemo") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# 创建表并写入
df = spark.read.json("s3://raw-data/events/")
df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .save("s3://my-bucket/warehouse/analytics/events")

# Append 写入
new_events.write \
    .format("delta") \
    .mode("append") \
    .save("s3://my-bucket/warehouse/analytics/events")

# 读取
df = spark.read.format("delta").load("s3://my-bucket/warehouse/analytics/events")

# 使用 DeltaTable API（更强大）
from delta.tables import DeltaTable
dt = DeltaTable.forPath(spark, "s3://my-bucket/warehouse/analytics/events")
# 或
dt = DeltaTable.forName(spark, "analytics.events")
```



### 2.3 Schema Evolution 操作

```python
# 方式一：写入时自动合并 schema
new_data_with_extra_col.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \  # 关键选项
    .save("s3://my-bucket/warehouse/analytics/events")

# 方式二：覆盖 schema（慎用，会改变整个表结构）
different_schema_data.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("s3://my-bucket/warehouse/analytics/events")
```

```sql
-- SQL 方式
ALTER TABLE analytics.events ADD COLUMNS (
    device_type STRING AFTER event_type,
    session_id STRING COMMENT 'User session identifier'
);

-- 重命名（需要启用 column mapping）
ALTER TABLE analytics.events
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

ALTER TABLE analytics.events RENAME COLUMN properties TO event_metadata;

-- 删除列（需要 column mapping mode = name）
ALTER TABLE analytics.events DROP COLUMN session_id;

-- 修改列类型（有限制：只能 widen，如 int → long）
ALTER TABLE analytics.events ALTER COLUMN event_id TYPE BIGINT;

-- 修改列注释
ALTER TABLE analytics.events ALTER COLUMN amount COMMENT 'Amount in USD';

-- 修改列 nullability
ALTER TABLE analytics.events ALTER COLUMN user_id DROP NOT NULL;
```

**Delta Column Mapping Mode 说明：**

| Mode | 行为 | 支持的 schema evolution |
|---|---|---|
| `none`（默认） | 按 column name + position 映射 | 只能 add columns |
| `name` | 按 column name 映射（加了内部 ID） | add、drop、rename |
| `id` | 按 column ID 映射（类似 Iceberg） | add、drop、rename（最灵活） |



### 2.4 Time Travel 操作

```sql
-- 按版本号查询
SELECT * FROM analytics.events VERSION AS OF 5;

-- 按时间查询
SELECT * FROM analytics.events TIMESTAMP AS OF '2024-06-15 10:00:00';

-- 查看表历史
DESCRIBE HISTORY analytics.events;
-- 返回：version, timestamp, operation, operationParameters, ...
```

```python
# PySpark API
df_v5 = spark.read.format("delta") \
    .option("versionAsOf", 5) \
    .load("s3://my-bucket/warehouse/analytics/events")

df_ts = spark.read.format("delta") \
    .option("timestampAsOf", "2024-06-15 10:00:00") \
    .load("s3://my-bucket/warehouse/analytics/events")

# 查看历史
dt = DeltaTable.forPath(spark, "s3://my-bucket/warehouse/analytics/events")
dt.history().show(truncate=False)
dt.history(10).show()   # 最近 10 条

# Rollback（通过 RESTORE）
spark.sql("RESTORE TABLE analytics.events TO VERSION AS OF 5")
# 或
spark.sql("RESTORE TABLE analytics.events TO TIMESTAMP AS OF '2024-06-15 10:00:00'")
```



### 2.5 Row-Level 操作（UPDATE / DELETE / MERGE）

```sql
-- DELETE
DELETE FROM analytics.events
WHERE event_time < '2023-01-01';

-- UPDATE
UPDATE analytics.events
SET amount = amount * 1.1
WHERE event_type = 'purchase';

-- MERGE INTO（upsert，Delta Lake 的明星功能）
MERGE INTO analytics.events AS target
USING staging_events AS source
ON target.event_id = source.event_id
WHEN MATCHED AND source.event_type = 'delete' THEN
    DELETE
WHEN MATCHED THEN
    UPDATE SET *          -- 更新所有列
WHEN NOT MATCHED THEN
    INSERT *;             -- 插入所有列
```

```python
# DeltaTable Python API（更灵活）
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, "s3://my-bucket/warehouse/analytics/events")

# DELETE
dt.delete("event_time < '2023-01-01'")

# UPDATE
dt.update(
    condition="event_type = 'purchase'",
    set={"amount": "amount * 1.1"}
)

# MERGE（Python builder pattern）
dt.alias("target").merge(
    source=staging_df.alias("source"),
    condition="target.event_id = source.event_id"
).whenMatchedUpdate(
    set={
        "event_type": "source.event_type",
        "amount": "source.amount"
    }
).whenNotMatchedInsert(
    values={
        "event_id": "source.event_id",
        "user_id": "source.user_id",
        "event_type": "source.event_type",
        "event_time": "source.event_time",
        "amount": "source.amount"
    }
).execute()
```

**Deletion Vectors（Delta 2.3+）：**

```sql
-- 启用 deletion vectors（MOR 模式的轻量级实现）
ALTER TABLE analytics.events
SET TBLPROPERTIES ('delta.enableDeletionVectors' = 'true');

-- 启用后，DELETE / UPDATE 不再 rewrite 整个文件
-- 而是写一个 deletion vector 文件（bitmap），标记哪些行被删除
-- 读取时自动跳过被标记的行
-- 需要定期 OPTIMIZE 来物理清除被标记的行
```



### 2.6 OPTIMIZE 与 Z-Ordering

```sql
-- 基础 OPTIMIZE（合并小文件）
OPTIMIZE analytics.events;

-- 只 optimize 特定 partition
OPTIMIZE analytics.events
WHERE event_date >= '2024-06-01';

-- Z-ORDER（多维排序，提升多列过滤的 data skipping 效率）
OPTIMIZE analytics.events
ZORDER BY (user_id, event_type);

-- Z-ORDER 的原理：
-- 将 (user_id, event_type) 映射到 Z 曲线（space-filling curve）
-- 排序后，相近的 (user_id, event_type) 组合被放到同一个文件中
-- 查询 WHERE user_id = 123 AND event_type = 'purchase' 时
-- 可以通过 file-level stats (min/max) 跳过大部分文件
```

**Liquid Clustering（Delta 3.0+ / Databricks）：**

```sql
-- 取代 Z-ORDER + 手动 partition 的新方案
CREATE TABLE analytics.events_v2 (...)
USING delta
CLUSTER BY (user_id, event_type);   -- 不再需要 PARTITIONED BY

-- Liquid Clustering 自动决定如何组织数据
-- 优势：
--   不需要手动运行 OPTIMIZE ZORDER
--   可以动态调整 clustering 维度（ALTER TABLE ... CLUSTER BY ...）
--   兼容增量写入，不需要全表 rewrite
```



### 2.7 VACUUM 与维护操作

```sql
-- VACUUM：物理删除不再被引用的文件
-- 默认保留 7 天内的文件（safety check）
VACUUM analytics.events;

-- 指定保留时间
VACUUM analytics.events RETAIN 168 HOURS;  -- 7 天

-- 危险操作：0 小时保留（生产中不推荐）
SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM analytics.events RETAIN 0 HOURS;

-- Dry run（只列出要删除的文件，不实际删除）
VACUUM analytics.events DRY RUN;
```

**VACUUM 的注意事项：**
- VACUUM 之后，早于保留时间的 time travel 将不可用（文件被物理删除了）
- 不要设置太短的保留时间，否则长时间运行的 query 可能因为文件被删除而失败
- VACUUM 不会删除 `_delta_log/` 中的 log 文件，log 的清理由 `logRetentionDuration` 控制



### 2.8 Change Data Feed（CDC 支持）

```sql
-- 启用 Change Data Feed
ALTER TABLE analytics.events
SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

-- 之后的每次 DML 操作都会记录变更（insert / update_preimage / update_postimage / delete）

-- 读取变更数据
SELECT * FROM table_changes('analytics.events', 5, 10);
-- 参数：table name, start version, end version

SELECT * FROM table_changes('analytics.events', '2024-06-01', '2024-06-15');
-- 参数：table name, start timestamp, end timestamp
```

```python
# PySpark API 读取 CDF
changes_df = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 5) \
    .option("endingVersion", 10) \
    .table("analytics.events")

# 返回的额外列：
# _change_type:  insert | update_preimage | update_postimage | delete
# _commit_version: 变更发生的版本号
# _commit_timestamp: 变更发生的时间

# Streaming 方式消费 CDF（实时 CDC 下游）
stream_df = spark.readStream.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 0) \
    .table("analytics.events")
```

**CDF vs Iceberg Incremental Read 对比：**

| 特性 | Delta CDF | Iceberg Incremental Read |
|---|---|---|
| 启用方式 | 需要显式开启 table property | 原生支持（通过 snapshot diff） |
| 变更类型 | insert / update_pre / update_post / delete | insert / delete（update = delete + insert） |
| 存储开销 | 额外的 `_change_data/` 目录存储变更数据 | 无额外存储，直接从 manifest status 推断 |
| Streaming 消费 | Spark Structured Streaming 原生支持 | Flink / Spark 均支持 |

---



## 3. Iceberg vs Delta Lake 操作对比速查

| 操作 | Iceberg | Delta Lake |
|---|---|---|
| **创建表** | `USING iceberg` | `USING delta` |
| **Append 写入** | `df.writeTo(table).append()` | `df.write.format("delta").mode("append")` |
| **Schema evolution** | `ALTER TABLE ... ADD/DROP/RENAME COLUMN` | `ALTER TABLE ...` + `mergeSchema` option |
| **Partition evolution** | `ALTER TABLE ADD PARTITION FIELD` | 不支持（需 rewrite 或用 Liquid Clustering） |
| **Hidden partitioning** | 原生支持（transform 函数） | 不支持（需 generated column workaround） |
| **Time travel** | `VERSION AS OF` / `TIMESTAMP AS OF` | 同 |
| **Rollback** | `CALL system.rollback_to_snapshot()` | `RESTORE TABLE ... TO VERSION AS OF` |
| **Compaction** | `CALL system.rewrite_data_files()` | `OPTIMIZE` |
| **Z-ordering** | Sort compaction（strategy='sort'） | `OPTIMIZE ... ZORDER BY` |
| **Expire snapshots** | `CALL system.expire_snapshots()` | 自动（`logRetentionDuration`） |
| **清理文件** | `CALL system.remove_orphan_files()` | `VACUUM` |
| **CDC / 增量读** | Incremental read（snapshot diff） | Change Data Feed |
| **Branch / Tag** | 原生支持（1.2+） | 不支持（Databricks 有 shallow clone） |
| **Row-level delete** | Positional / equality delete files | Deletion vectors |
| **Merge into** | 支持（Spark 3.x） | 支持（DeltaTable API 更成熟） |

---



## 4. 生产环境最佳实践

### 4.1 文件大小管理

```text
目标文件大小：256MB - 1GB（根据查询模式调整）
- OLAP 重度聚合查询 → 偏大（512MB - 1GB）减少文件数
- 低延迟 point lookup → 偏小（128MB - 256MB）减少单文件扫描量
- Streaming 写入 → 小文件不可避免，依赖定期 compaction

Iceberg 配置：
  write.target-file-size-bytes = 536870912  (512MB)

Delta 配置：
  delta.targetFileSize = 536870912
  spark.databricks.delta.optimizeWrite.enabled = true  (自动合并小文件)
```

### 4.2 Compaction 调度策略

```text
策略一：定时调度（最常见）
  - 每小时跑一次 lightweight compaction（只处理最近写入的 partition）
  - 每天跑一次 full compaction（处理所有 partition）

策略二：阈值触发
  - 监控每个 partition 的文件数量
  - 当小文件数超过阈值（如 50 个）时触发 compaction

策略三：Auto compaction（Delta on Databricks）
  - delta.autoOptimize.autoCompact = true
  - 写入后自动触发后台 compaction
```

### 4.3 Table Maintenance DAG（完整示例）

```python
# Airflow DAG 伪代码 — Iceberg 日常维护
from airflow import DAG
from airflow.providers.apache.spark.operators.spark_sql import SparkSqlOperator

with DAG('iceberg_maintenance', schedule='0 3 * * *') as dag:  # 每天凌晨 3 点

    compact = SparkSqlOperator(
        task_id='compact_recent_partitions',
        sql="""
            CALL iceberg_catalog.system.rewrite_data_files(
                table => 'analytics.events',
                where => 'event_time >= current_date() - INTERVAL 2 DAYS',
                options => map(
                    'target-file-size-bytes', '536870912',
                    'min-file-size-bytes', '67108864'
                )
            )
        """
    )

    rewrite_manifests = SparkSqlOperator(
        task_id='rewrite_manifests',
        sql="CALL iceberg_catalog.system.rewrite_manifests('analytics.events')"
    )

    expire = SparkSqlOperator(
        task_id='expire_old_snapshots',
        sql="""
            CALL iceberg_catalog.system.expire_snapshots(
                'analytics.events',
                current_timestamp() - INTERVAL 7 DAYS
            )
        """
    )

    remove_orphans = SparkSqlOperator(
        task_id='remove_orphan_files',
        sql="""
            CALL iceberg_catalog.system.remove_orphan_files(
                table => 'analytics.events',
                older_than => current_timestamp() - INTERVAL 3 DAYS
            )
        """
    )

    compact >> rewrite_manifests >> expire >> remove_orphans
```

### 4.4 监控指标

```text
需要监控的关键指标：
- 每个 partition 的文件数量（过多 → 需要 compaction）
- 平均文件大小（过小 → 需要 compaction）
- Manifest 文件数量（过多 → 需要 rewrite_manifests）
- Snapshot 数量（过多 → 需要 expire）
- Orphan files 数量（占用存储成本）
- Query planning time（metadata 过多会导致 planning 变慢）
- Commit retry 次数（OCC 冲突频率）

监控方式：
  Iceberg: SELECT * FROM table.files / table.manifests / table.snapshots
  Delta:   DESCRIBE DETAIL table / DESCRIBE HISTORY table
```

---



## 5. 面试高频追问与回答

### Q: Iceberg 的 optimistic concurrency control 具体怎么处理冲突？

当两个 writer 同时基于 snapshot S0 开始工作时，假设 Writer A 先完成并成功将 metadata pointer 从 S0 更新到 S1。Writer B 完成后也尝试更新 pointer，发现当前已经是 S1 而不是 S0。此时 Iceberg 会做 conflict detection：检查 B 修改的文件集合和 A 修改的文件集合是否有交集。如果没有交集（比如 A 写 partition=2024-06-14，B 写 partition=2024-06-15），B 可以基于 S1 安全 retry。如果有交集，则 commit 失败，需要应用层重新处理。

### Q: Delta Lake 的 checkpoint 和 Iceberg 的 manifest list 有什么区别？

Delta 的 checkpoint 是把所有 commit 合并成一个 Parquet 文件来加速 log replay，本质上是**优化读取性能的快照**。Iceberg 的 manifest list 是每个 snapshot 的组成部分，它列出当前 snapshot 对应的所有 manifest files 及其 partition summary，是**数据组织的核心结构**。Delta checkpoint 是 compaction of log entries，Iceberg manifest list 是 index of data files。

### Q: 什么时候用 COW，什么时候用 MOR？

COW 适合**读多写少**的场景（如 nightly batch update + 大量 OLAP query），因为写入时的 rewrite 成本被分摊到大量读取中。MOR 适合**写多读少**或**需要低延迟写入**的场景（如 streaming CDC 写入），因为写入只需要追加小的 delete file。但 MOR 的读取性能会随着未合并的 delete files 增加而下降，所以需要定期 compaction。

### Q: Hidden partitioning 为什么重要？

Hidden partitioning 解决的是**用户 query 和物理 partition 之间的耦合问题**。传统 Hive 分区要求用户在 WHERE 中精确引用分区列（`WHERE year=2024 AND month=6`），否则 partition pruning 不生效。Iceberg 的 hidden partitioning 让引擎自动从 filter predicate 推断出要访问哪些 partition（用户写 `WHERE event_time > '2024-06-15'`，引擎自动推断出对应的 `day` partition）。这降低了用户理解物理布局的心智负担，也使 partition evolution 成为可能——换分区策略不需要改 query。